# KNN Regression on Retail Sales Data

**Objective:** Predict **Total Sales** (continuous target) for retail customers using K-Nearest Neighbors regression.

**Pipeline:** Load Data → EDA → Feature Engineering → Train/Test Split → Hyperparameter Tuning → Evaluation → Prediction Analysis

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from retail_data import generate_retail_dataset

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)
RANDOM_STATE = 42

## 1. Load Retail Dataset

In [ ]:
df = generate_retail_dataset(n_samples=2000, random_state=RANDOM_STATE)
print(f'Shape: {df.shape}')
df.head()

## 2. Exploratory Data Analysis

In [ ]:
print('Target variable: Total_Sales')
print(df['Total_Sales'].describe())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df['Total_Sales'], kde=True, ax=axes[0])
axes[0].set_title('Total Sales Distribution')
sns.boxplot(data=df, x='Customer_Segment', y='Total_Sales', ax=axes[1], palette='Set2')
axes[1].set_title('Total Sales by Customer Segment')
plt.tight_layout()
plt.show()

In [ ]:
num_features = ['Age', 'Annual_Income', 'Spending_Score', 'Num_Purchases', 'Avg_Transaction_Value']
corr_with_target = df[num_features + ['Total_Sales']].corr()['Total_Sales'].drop('Total_Sales').sort_values(ascending=False)
print('Correlation with Total_Sales:')
print(corr_with_target)

sns.barplot(x=corr_with_target.values, y=corr_with_target.index, palette='Blues_d')
plt.title('Feature Correlation with Total Sales')
plt.xlabel('Correlation')
plt.show()

## 3. Feature Engineering & Train/Test Split

In [ ]:
cat_features = ['Region', 'Product_Category', 'Purchase_Channel']
target = 'Total_Sales'

X = df[num_features + cat_features]
y = df[target]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(f'Train: {X_train.shape[0]}, Test: {X_test.shape[0]}')

In [ ]:
preprocessor = ColumnTransformer([
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_features)
])

knn_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', KNeighborsRegressor())
])

## 4. Hyperparameter Tuning (K, Weights, Distance Metric)

In [ ]:
param_grid = {
    'regressor__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
    'regressor__weights': ['uniform', 'distance'],
    'regressor__metric': ['euclidean', 'manhattan']
}

grid_search = GridSearchCV(
    knn_pipeline, param_grid, cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1, verbose=1
)
grid_search.fit(X_train, y_train)

print(f'Best params: {grid_search.best_params_}')
print(f'Best CV MAE: {-grid_search.best_score_:.2f}')

In [ ]:
results = pd.DataFrame(grid_search.cv_results_)
k_results = results.groupby('param_regressor__n_neighbors')['mean_test_score'].mean().reset_index()
k_results['mae'] = -k_results['mean_test_score']

plt.figure(figsize=(10, 5))
plt.plot(k_results['param_regressor__n_neighbors'], k_results['mae'], 'bo-')
plt.xlabel('K (n_neighbors)')
plt.ylabel('CV MAE')
plt.title('KNN Regression: MAE vs K')
plt.show()

## 5. Evaluate Best Model on Test Set

In [ ]:
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print('=== Test Set Performance ===')
print(f'MAE:  ${mae:,.2f}')
print(f'RMSE: ${rmse:,.2f}')
print(f'R²:   {r2:.4f}')

## 6. Visualization: Predicted vs Actual

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y_test, y_pred, alpha=0.5, edgecolors='k', linewidth=0.3)
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0].set_xlabel('Actual Total Sales ($)')
axes[0].set_ylabel('Predicted Total Sales ($)')
axes[0].set_title('Predicted vs Actual')

residuals = y_test - y_pred
sns.histplot(residuals, kde=True, ax=axes[1])
axes[1].axvline(0, color='r', linestyle='--')
axes[1].set_title('Residual Distribution')
axes[1].set_xlabel('Residual ($)')
plt.tight_layout()
plt.show()

In [ ]:
sample_results = pd.DataFrame({
    'Actual': y_test.values,
    'Predicted': y_pred,
    'Error': y_test.values - y_pred,
    'Abs_Error_Pct': (np.abs(y_test.values - y_pred) / y_test.values * 100).round(1)
}).head(15)
sample_results

## 7. Cross-Validation Stability

In [ ]:
cv_scores = cross_val_score(best_model, X, y, cv=5, scoring='r2', n_jobs=-1)
print(f'CV R² scores: {cv_scores.round(4)}')
print(f'Mean CV R²: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

## 8. Make Predictions on New Customers

In [ ]:
new_customers = pd.DataFrame({
    'Age': [28, 55, 42],
    'Annual_Income': [45000, 95000, 70000],
    'Spending_Score': [75, 30, 60],
    'Num_Purchases': [20, 8, 15],
    'Avg_Transaction_Value': [45.0, 120.0, 85.0],
    'Region': ['North', 'West', 'East'],
    'Product_Category': ['Electronics', 'Furniture', 'Clothing'],
    'Purchase_Channel': ['Online', 'In-Store', 'Mobile']
})

new_predictions = best_model.predict(new_customers)
new_customers['Predicted_Total_Sales'] = new_predictions.round(2)
new_customers

## 9. Conclusions

- **KNN Regression** predicts total sales by averaging (or distance-weighted averaging) the sales of the K most similar customers.
- Feature scaling is critical — KNN is distance-based.
- Tune `n_neighbors`, `weights`, and `metric` via cross-validation.
- **Use case:** Revenue forecasting, inventory planning, and identifying high-value customer profiles.
- **Limitation:** Slow inference on large datasets; sensitive to irrelevant features and curse of dimensionality.